# Italy — Preprocessing notebook

**Period:** 2024-01-01 → 2024-06-30

**Raw inputs** (`Data/Italy/Raw/`):
- `Train_operation_data.csv` — ~600 MB, ~2.6 M stop rows
- `Train_station_locations_data.csv` — station coordinates
- `Adjacent_railway_stations_mileage_data.csv` — route sequences with cumulative mileage
- `Train_fault_information.csv` — line-level fault logs (latin-1)

**Outputs** (`Data/Italy/processed/`): the five standardized CSVs of `utils.SCHEMA`.

Heavy logic lives in `preprocess/lib_italy.py`; this notebook orchestrates and renders inspection plots for every transformation step.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().resolve()
while ROOT.name and not (ROOT / "utils.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "preprocess"))

import lib_italy as lib
from utils import SCHEMA, save_csv

RAW = ROOT / "Data" / "Italy" / "Raw"
OUT = ROOT / "Data" / "Italy" / "processed"
FIG = ROOT / "preprocess" / "figures" / "italy"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
np.random.seed(42)

def save_fig(fig, name: str) -> Path:
    p = FIG / f"{name}.png"
    fig.savefig(p, bbox_inches="tight")
    return p

## Step 1 — Stream-load operations

The 600 MB operations CSV is loaded in 250 k-row chunks. Each chunk is filtered to the date window before being kept, so peak RAM stays well below the file size. `lib.load_operations_chunked` returns the concatenated frame **and** a per-chunk telemetry table that we use for the streaming-progress plots.

In [ ]:
ops, chunk_stats = lib.load_operations_chunked(RAW / "Train_operation_data.csv")
print(f"Operations rows kept: {len(ops):,}")
chunk_stats.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].bar(chunk_stats["chunk_idx"], chunk_stats["raw_rows"], color="#90A4AE", label="raw")
axes[0].bar(chunk_stats["chunk_idx"], chunk_stats["kept_rows"], color="#1E88E5", label="kept")
axes[0].set_title("Rows per chunk")
axes[0].set_xlabel("chunk #"); axes[0].set_ylabel("rows"); axes[0].legend()
axes[1].plot(chunk_stats["chunk_idx"], chunk_stats["cum_kept"], marker="o", color="#1E88E5")
axes[1].set_title("Cumulative kept rows"); axes[1].set_xlabel("chunk #")
axes[2].hist(ops["date"], bins=24, color="#1E88E5", edgecolor="white")
axes[2].set_title("Date distribution"); axes[2].set_xlabel("date")
fig.suptitle("Step 1 — Streaming load", fontweight="bold")
save_fig(fig, "step_01_streaming_load"); plt.show()

## Step 2 — Delay sentinel cleaning

Italian rail delay columns contain two non-numeric sentinels: `'N'` (first stop, no inbound arrival) and `'S'` (Soppresso = cancelled). `lib.clean_delay_column` returns the parsed minutes plus a boolean cancellation mask. The stacked bar below shows the proportion of each value class — useful for confirming the sentinel proportions are sane.

In [ ]:
def classify(series):
    s = series.astype(str).str.strip()
    return pd.Series({
        "N (first stop)": int(s.eq("N").sum()),
        "S (cancelled)":  int(s.eq("S").sum()),
        "numeric":        int(pd.to_numeric(s, errors="coerce").notna().sum()),
    })

raw_breakdown = pd.DataFrame({
    "arrival_delay":   classify(ops["arrival_delay"]),
    "departure_delay": classify(ops["departure_delay"]),
})

ops["arrival_delay"],   arr_c = lib.clean_delay_column(ops["arrival_delay"])
ops["departure_delay"], dep_c = lib.clean_delay_column(ops["departure_delay"])
ops["cancelled"] = arr_c | dep_c
ops["station_name"] = ops["station_name"].astype(str).str.strip().str.upper()
raw_breakdown

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
raw_breakdown.T.plot(kind="bar", stacked=True, ax=ax,
                      color=["#FFA000", "#E53935", "#1E88E5"], edgecolor="white")
ax.set_title("Step 2 — Delay value breakdown before cleaning", fontweight="bold")
ax.set_ylabel("row count"); ax.set_xlabel("")
ax.legend(title="value class")
save_fig(fig, "step_02_sentinel_breakdown"); plt.show()

## Step 3 — Delay distribution after cleaning

Histogram of `arrival_delay` (minutes), log-scaled y-axis to make the long tail readable. The vertical dashed line marks the 5-minute disruption threshold (`utils.DELAY_THRESHOLD_MIN`). A KDE per train class shows whether disruption-prone classes have a heavier right tail.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
delays = ops["arrival_delay"].dropna()
axes[0].hist(delays.clip(-10, 90), bins=80, color="#1E88E5", edgecolor="white")
axes[0].set_yscale("log")
axes[0].axvline(5, ls="--", color="#E53935", label="5-min threshold")
axes[0].set_title("arrival_delay histogram (log y)"); axes[0].set_xlabel("minutes")
axes[0].legend()

sample = ops.dropna(subset=["arrival_delay", "train_class"]).sample(
    min(200_000, len(ops)), random_state=42)
for cls, sub in sample.groupby("train_class"):
    if len(sub) < 200:
        continue
    sns.kdeplot(sub["arrival_delay"].clip(-10, 60), ax=axes[1],
                label=str(cls), bw_adjust=1.2)
axes[1].axvline(5, ls="--", color="#E53935")
axes[1].set_title("KDE by train class"); axes[1].set_xlabel("minutes")
axes[1].legend(title="train class", fontsize=8, loc="upper right")
fig.suptitle("Step 3 — Delay distribution after cleaning", fontweight="bold")
save_fig(fig, "step_03_delay_distribution"); plt.show()

## Step 4 — Weather text → ordinal severity

Italy stores weather as free-text Italian labels (`cloudy`, `light rain`, `heavy snow`, …). `lib.WEATHER_SEVERITY_MAP` projects those to a 0–4 ordinal scale. The heatmap shows the projection, the bar shows the resulting severity counts after applying it to all stops.

In [ ]:
ops["weather_severity"] = lib.map_weather_severity(ops["weather"])

mapping_df = (
    pd.Series(lib.WEATHER_SEVERITY_MAP, name="severity")
      .reset_index().rename(columns={"index": "label"})
)
mapping_pivot = mapping_df.pivot_table(
    index="label", columns="severity", values="severity", aggfunc="size", fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.heatmap(mapping_pivot, cmap="Reds", cbar=False, annot=True, ax=axes[0])
axes[0].set_title("Raw text → severity mapping")

sev_counts = ops["weather_severity"].value_counts().sort_index()
axes[1].bar(sev_counts.index.astype(str), sev_counts.values,
            color=sns.color_palette("Reds", n_colors=5))
axes[1].set_title("Severity ordinal counts"); axes[1].set_xlabel("severity 0–4")
axes[1].set_ylabel("row count")
fig.suptitle("Step 4 — Weather severity", fontweight="bold")
save_fig(fig, "step_04_weather_severity"); plt.show()

## Step 5 — Station coordinates and risk map

We build `nodes_station.csv` and visualise the network geographically. Markers are coloured by `avg_historical_delay` — chronically late stations show up immediately.

In [ ]:
stations_master = pd.read_csv(RAW / "Train_station_locations_data.csv", encoding="latin1")
stations_master["name"] = stations_master["name"].str.strip().str.upper()

mileage = pd.read_csv(RAW / "Adjacent_railway_stations_mileage_data.csv")
mileage["station_name"] = mileage["station_name"].str.strip().str.upper()
mileage["distance"] = pd.to_numeric(
    mileage["distance"].astype(str).str.replace(",", "."), errors="coerce")

nodes_station = lib.build_nodes_station(ops, stations_master, mileage)
print(f"{len(nodes_station):,} stations | missing coords: {nodes_station['lat'].isna().sum()}")
nodes_station.head()

In [ ]:
geo = nodes_station.dropna(subset=["lat", "lon"])
fig, ax = plt.subplots(figsize=(8, 9))
sc = ax.scatter(geo["lon"], geo["lat"],
                c=geo["avg_historical_delay"].clip(0, 20),
                cmap="Reds", s=18, edgecolor="k", linewidth=0.2)
plt.colorbar(sc, ax=ax, label="avg historical delay (min)")
ax.set_title("Step 5 — Italy station map (colour = avg delay)", fontweight="bold")
ax.set_xlabel("lon"); ax.set_ylabel("lat")
save_fig(fig, "step_05_station_map"); plt.show()

## Step 6 — Mileage graph: degree distribution and route lengths

The adjacency we ship comes from consecutive station pairs in the per-route mileage table. Two diagnostics:
- **Histogram of station degree** — most stations are simple line nodes (degree 2); hubs are the right tail.
- **CCDF of route lengths** — how many stations a typical service visits.

In [ ]:
sid_map = {n: f"IT_{n.replace(' ', '_')}" for n in ops['station_name'].dropna().unique()}
edges_adjacent = lib.build_edges_adjacent(mileage, sid_map)
print(f"edges_adjacent: {len(edges_adjacent):,} directed edges")

route_lengths = mileage.groupby("train_id")["station_order"].max().dropna().astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].hist(nodes_station["degree"].clip(upper=20), bins=20, color="#1E88E5", edgecolor="white")
axes[0].set_title("Degree distribution"); axes[0].set_xlabel("degree (clipped at 20)")

x = np.sort(route_lengths.values)
y = 1.0 - np.arange(1, len(x) + 1) / len(x)
axes[1].step(x, y, where="post", color="#1E88E5")
axes[1].set_xscale("log"); axes[1].set_yscale("log")
axes[1].set_title("CCDF of route lengths"); axes[1].set_xlabel("# stops"); axes[1].set_ylabel("P(X ≥ x)")
fig.suptitle("Step 6 — Mileage graph topology", fontweight="bold")
save_fig(fig, "step_06_topology"); plt.show()

## Step 7 — Service labelling

For each `(train_id, date)` we set `is_disrupted = 1` if any stop was late > 5 min or cancelled. We plot disruption rate by `train_class_code` and by month — the train-class effect should be visible (commuter ≠ high-speed).

In [ ]:
nodes_service = lib.build_nodes_service(ops)
print(f"services: {len(nodes_service):,} | disruption rate = {nodes_service['is_disrupted'].mean()*100:.2f}%")

by_class = (
    nodes_service.groupby("train_class_code")["is_disrupted"].mean().reset_index()
)
by_month = (
    nodes_service.assign(month=pd.to_datetime(nodes_service["date"]).dt.month)
                  .groupby("month")["is_disrupted"].mean().reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.barplot(data=by_class, x="train_class_code", y="is_disrupted",
            ax=axes[0], color="#1E88E5")
axes[0].set_title("Disruption rate by train class"); axes[0].set_ylim(0, 1)

sns.lineplot(data=by_month, x="month", y="is_disrupted", ax=axes[1],
             marker="o", color="#E91E63")
axes[1].set_title("Disruption rate by month"); axes[1].set_xticks(range(1, 7))
axes[1].set_ylim(0, 1)
fig.suptitle("Step 7 — Service-level disruption rates", fontweight="bold")
save_fig(fig, "step_07_service_labelling"); plt.show()

## Step 8 — Fault logs

The Italian fault file is line-level free text (`station_id` is `NaN`). We persist it as-is for the Knowledge Graph layer to consume; here we show a top-30 keyword bar of `delay_reason` and the daily fault count timeline.

In [ ]:
faults = pd.read_csv(RAW / "Train_fault_information.csv", encoding="latin1")
faults["date"] = pd.to_datetime(faults["date"], format="%Y/%m/%d", errors="coerce")
nodes_fault = lib.build_nodes_fault(faults)
print(f"fault rows kept: {len(nodes_fault):,}")

import re
tokens = (nodes_fault["description"].dropna().astype(str)
          .str.lower().str.findall(r"[a-zà-ù]{4,}"))
all_tokens = pd.Series([t for lst in tokens for t in lst])
top_words = all_tokens.value_counts().head(30)

daily = nodes_fault.groupby(pd.to_datetime(nodes_fault["date"]).dt.date).size()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].barh(top_words.index[::-1], top_words.values[::-1], color="#FFA000")
axes[0].set_title("Top-30 fault keywords"); axes[0].set_xlabel("count")
axes[1].plot(daily.index, daily.values, color="#FFA000")
axes[1].set_title("Daily fault count"); axes[1].set_xlabel("date")
fig.suptitle("Step 8 — Fault logs", fontweight="bold")
save_fig(fig, "step_08_faults"); plt.show()

## Step 9 — Persist all five standardized CSVs

Final validation: row counts, schema match against `utils.SCHEMA`, sample rows. After this cell, `Data/Italy/processed/` is ready for the unified preprocessor.

In [ ]:
edges_stops_at = lib.build_edges_stops_at(ops, sid_map)

save_csv(nodes_station,  OUT / "nodes_station.csv",  "nodes_station")
save_csv(nodes_service,  OUT / "nodes_service.csv",  "nodes_service")
save_csv(edges_stops_at, OUT / "edges_stops_at.csv", "edges_stops_at")
save_csv(edges_adjacent, OUT / "edges_adjacent.csv", "edges_adjacent")
save_csv(nodes_fault,    OUT / "nodes_fault.csv",    "nodes_fault")

summary = pd.DataFrame([
    {"file": "nodes_station.csv",  "rows": len(nodes_station),  "cols": len(SCHEMA["nodes_station"])},
    {"file": "nodes_service.csv",  "rows": len(nodes_service),  "cols": len(SCHEMA["nodes_service"])},
    {"file": "edges_stops_at.csv", "rows": len(edges_stops_at), "cols": len(SCHEMA["edges_stops_at"])},
    {"file": "edges_adjacent.csv", "rows": len(edges_adjacent), "cols": len(SCHEMA["edges_adjacent"])},
    {"file": "nodes_fault.csv",    "rows": len(nodes_fault),    "cols": len(SCHEMA["nodes_fault"])},
])
summary

In [ ]:
for name, df in [("nodes_station", nodes_station), ("nodes_service", nodes_service),
                  ("edges_stops_at", edges_stops_at), ("edges_adjacent", edges_adjacent),
                  ("nodes_fault", nodes_fault)]:
    expected = set(SCHEMA[name])
    actual   = set(df.columns)
    assert expected.issubset(actual), f"{name}: missing {expected - actual}"
print("✓ Schema validation passed for all five outputs.")

## Closing summary

Italy preprocessing complete. The CSVs in `Data/Italy/processed/` plus the figures in `preprocess/figures/italy/` are the artefacts the unified stop-level pipeline (`stop_level/preprocess_stops.py`) will consume.